# Complaint Ranking

## Import Libraries

In [42]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

## Load Dataset

In [43]:
df = pd.read_csv("../data/processed/feature_engineered_complaints.csv")

In [44]:
df.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,public_safety_risk,recurrence_indicator,severity,vulnerable_population_ratio,complaint_age_days,is_active_complaint,ward_complaint_load
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,Medium,No,High,0.169565,0,False,25
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Critical,No,Medium,0.200000,0,False,8
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,Medium,No,Medium,0.063158,0,False,67
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,Low,No,Low,0.285714,0,False,10
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Low,No,High,0.150000,0,False,7


In [45]:
df.shape

(8953, 16)

## Required Columns

In [46]:
required_columns = [
    "complaint_id",
    "category",
    "sub_category",
    "ward",
    "reported_date",
    "status",
    "people_affected",
    "vulnerable_population",
    "public_safety_risk",
    "recurrence_indicator"
]

In [47]:
df["severity"] = df["severity"].astype(str).str.strip().str.title()
df["status"] = df["status"].astype(str).str.strip().str.title()

In [48]:
numeric_columns = [
    "complaint_age_days",
    "people_affected",
    "vulnerable_population_ratio",
    "public_safety_risk",
    "recurrence_indicator"
]

In [49]:
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_columns] = df[numeric_columns].fillna(0)

## Create a provisional priority target

In [50]:
severity_scores = {
    "Low": 25,
    "Medium": 50,
    "High": 75,
    "Critical": 100
}

In [51]:
status_scores = {
    "Registered": 100,
    "Verified": 80,
    "Assigned": 60,
    "In Progress": 40
}

In [52]:
df["severity_score"] = df["severity"].map(severity_scores)

In [53]:
df["people_affected"] = pd.to_numeric(
    df["people_affected"], errors="coerce"
).fillna(0).clip(lower=0)

In [54]:
df["age_score"] = (
    df["complaint_age_days"].clip(0, 30) / 30 * 100
)

max_people = max(df["people_affected"].max(), 1)

df["people_score"] = (
    np.log1p(df["people_affected"].clip(lower=0))
    / np.log1p(max_people) * 100
)

df["vulnerability_score"] = (
    df["vulnerable_population_ratio"].clip(0, 1) * 100
)

df["safety_score"] = (
    df["public_safety_risk"].clip(0, 1) * 100
)

df["recurrence_score"] = (
    df["recurrence_indicator"].clip(0, 1) * 100
)

df["status_score"] = df["status"].map(status_scores).fillna(0)

In [55]:
df["priority_target"] = (
    df["severity_score"] * 0.30
    + df["safety_score"] * 0.20
    + df["people_score"] * 0.15
    + df["vulnerability_score"] * 0.15
    + df["age_score"] * 0.10
    + df["status_score"] * 0.05
    + df["recurrence_score"] * 0.05
)

In [56]:
print(df[["complaint_id", "priority_target"]].head())

  complaint_id  priority_target
0      C000001        32.993902
1      C000002        24.227032
2      C000003        23.620024
3      C000004        14.823416
4      C000005        31.979276


## Select features

In [57]:
categorical_features = [
    "severity",
    "status"
]

numerical_features = [
    "public_safety_risk",
    "people_affected",
    "vulnerable_population_ratio",
    "complaint_age_days"
]

features = categorical_features + numerical_features

X = df[features].copy()
Y = df["priority_target"].copy()

for col in categorical_features:
    X[col] = X[col].fillna("Unknown").astype(str)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (8953, 6)
Y shape: (8953,)


## Split data and train the ML model

In [58]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

priority_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

priority_model.fit(X_train, Y_train)

Y_pred = priority_model.predict(X_test)

print("MAE:", mean_absolute_error(Y_test, Y_pred))
print("R² score:", r2_score(Y_test, Y_pred))

MAE: 0.12405695598412503
R² score: 0.9980569673455209


## Predict scores and rank active complaints

In [59]:
df["predicted_priority_score"] = priority_model.predict(X)

completed_statuses = ["Resolved", "Closed"]

ranked_df = df[
    ~df["status"].isin(completed_statuses)
].copy()

ranked_df = ranked_df.sort_values(
    by="predicted_priority_score",
    ascending=False
).reset_index(drop=True)

ranked_df.insert(
    0,
    "priority_rank",
    range(1, len(ranked_df) + 1)
)

ranked_df["priority_level"] = pd.cut(
    ranked_df["predicted_priority_score"],
    bins=[-np.inf, 40, 60, 80, np.inf],
    labels=["Low", "Medium", "High", "Urgent"]
)

print(
    ranked_df[
        [
            "priority_rank",
            "complaint_id",
            "severity",
            "predicted_priority_score",
            "priority_level",
            "status"
        ]
    ].head(10)
)

   priority_rank complaint_id  severity  predicted_priority_score  \
0              1      C006349  Critical                 59.517283   
1              2      C006779  Critical                 59.492686   
2              3      C006364  Critical                 58.030602   
3              4      C003427  Critical                 57.658629   
4              5      C006306  Critical                 57.234933   
5              6      C007034  Critical                 57.102615   
6              7      C006355  Critical                 57.060701   
7              8      C008159  Critical                 56.517567   
8              9      C003418  Critical                 56.431353   
9             10      C003389  Critical                 56.298092   

  priority_level       status  
0         Medium     Assigned  
1         Medium     Verified  
2         Medium     Assigned  
3         Medium     Assigned  
4         Medium     Assigned  
5         Medium  In Progress  
6         Medium

## Save the ranked queue

In [60]:
ranked_df.to_csv(
    "../data/processed/rank_complaints.csv",
    index= False
)

## Final validation

In [61]:
check_df = pd.read_csv("../data/processed/rank_complaints.csv")
print(check_df.columns.tolist())

['priority_rank', 'complaint_id', 'citizen_id', 'category', 'sub_category', 'ward', 'reported_date', 'status', 'people_affected', 'vulnerable_population', 'public_safety_risk', 'recurrence_indicator', 'severity', 'vulnerable_population_ratio', 'complaint_age_days', 'is_active_complaint', 'ward_complaint_load', 'severity_score', 'age_score', 'people_score', 'vulnerability_score', 'safety_score', 'recurrence_score', 'status_score', 'priority_target', 'predicted_priority_score', 'priority_level']


In [62]:


print("Ranked complaints shape:", check_df.shape)
print("\nTop 5 ranked complaints:")
print(check_df[
    ["priority_rank", "complaint_id",
     "predicted_priority_score", "priority_level"]
].head())

Ranked complaints shape: (1306, 27)

Top 5 ranked complaints:
   priority_rank complaint_id  predicted_priority_score priority_level
0              1      C006349                 59.517283         Medium
1              2      C006779                 59.492686         Medium
2              3      C006364                 58.030602         Medium
3              4      C003427                 57.658629         Medium
4              5      C006306                 57.234933         Medium
